# 실습 ③ 다중 벡터 검색기 + 최종 체인 — 연습본


색인 카드(요약문)를 **서랍**에 꽂고, 원본을 **서가**에 보관해 연결합니다. 오늘 수업의 마지막 조각입니다.

---

## 오늘 만드는 것

```
질문 ──▶ 요약으로 검색 ──▶ 원본 꺼내기 ──▶ 멀티모달 LLM ──▶ 답변
         (색인 카드)       (진짜 책)        (눈 달린 모델)
```

## 진행 방법

1. 셀을 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`)
2. 코드 셀의 **주석을 따라 코드를 직접 작성**합니다 (`# TODO` 는 핵심 포인트)
3. 각 단계의 **확인 포인트**를 통과하면 다음으로 넘어갑니다

## ⚠️ 마지막 8번 셀을 꼭 하세요

**Before/After 비교**는 과제 채점에서 7점이고, 포트폴리오에서 가장 강력한 자료입니다. 시간이 모자라면 9번(번역 체인)을 건너뛰고 8번을 하세요.

## 0. 결과 복원

지금까지 만든 모든 재료를 한 번에 불러옵니다.

> ⚠️ `FileNotFoundError`가 나면 앞 교시의 저장 셀을 실행하지 않은 것입니다.

In [ ]:
# ── 0. 결과 복원 ──────────────────────────────────────────
# 1) 필요한 모듈 import: os, re, time, uuid, pickle, pathlib.Path
# 2) python-dotenv 의 load_dotenv import

# 3) .env 가 있는 폴더를 프로젝트 루트로 자동 탐색
#    - Path.cwd() 와 그 상위 폴더(parents)를 차례로 확인해 (p / ".env") 가 존재하는 첫 경로를 PROJECT_ROOT 로 지정
#    - 찾지 못하면 Path.cwd() 를 사용
#    - PROJECT_ROOT 출력 → c:\workspaces\multimodal-rag 이면 정상

# 4) PROJECT_ROOT 의 .env 파일 로드 (load_dotenv)
# 5) OPENAI_API_KEY 가 없으면 ValueError 발생시키기

# 6) processed 경로 = PROJECT_ROOT / "data" / "processed"

# 7) 추출 결과 복원: processed/"extracted.pkl" 을 pickle 로 읽기 ("rb")
#    - texts, tables, texts_2k_token 꺼내기

# 8) 요약 결과 복원: processed/"summaries.pkl" 을 pickle 로 읽기
#    - text_summaries, table_summaries, image_summaries, img_base64_list 꺼내기

# 9) "복원 완료" 와 함께 텍스트·표·이미지의 원본 수 / 요약 수 출력

> ✅ **확인 포인트:** 텍스트·표·이미지의 원본 수와 요약 수가 각각 같아야 합니다. (표는 0건일 수 있습니다)

## 1. 다중 벡터 검색기의 구조

> **비유:** **도서관의 색인 카드**입니다. 사서는 책을 다 읽고 찾지 않습니다. 요약이 적힌 **색인 카드로 검색**하고, 찾으면 **서가에서 진짜 책을 꺼내** 건네줍니다.

| 구성 요소 | 비유 | 들어가는 것 | 역할 |
| --- | --- | --- | --- |
| `vectorstore` (Chroma) | 색인 카드 서랍 | 요약문 임베딩 | **검색** |
| `docstore` (InMemoryStore) | 서가 | 원본 텍스트·표·Base64 이미지 | **전달** |
| `id_key` | 청구기호 | `doc_id` | 둘을 **연결** |

오늘 배우는 것의 핵심은 한 줄입니다.

> **요약과 원본에 같은 번호표를 붙인다.**

## 2. 검색기 만들기

아래 셀의 주석을 따라 코드를 작성하세요.

> `InMemoryStore`는 메모리에만 저장됩니다. 커널을 재시작하면 사라지므로 실제 서비스에서는 파일이나 Redis를 씁니다.

In [ ]:
# ── 2. 검색기 만들기 ──────────────────────────────────────
# langchain 1.x 에서 두 클래스의 위치가 바뀌었습니다.
# 어느 버전을 설치했든 동작하도록 두 경로를 모두 시도합니다. (try / except ImportError)

# 1) MultiVectorRetriever import
#    - langchain 1.x : langchain_classic.retrievers.multi_vector
#    - langchain 0.3 : langchain.retrievers.multi_vector

# 2) InMemoryStore import
#    - langchain 1.x : langchain_core.stores
#    - langchain 0.3 : langchain.storage

# 3) Chroma(langchain_chroma), OpenAIEmbeddings(langchain_openai) import

# 4) 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
#    - collection_name="multimodal_rag"
#    - embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")

# 5) 원본이 들어갈 저장소 (= 서가) : InMemoryStore()
# 6) 연결 키(= 청구기호) : id_key = "doc_id"

# 7) MultiVectorRetriever(vectorstore=..., docstore=..., id_key=...) 생성
# 8) "검색기 생성 완료" 출력

## 3. 요약과 원본을 같은 ID로 짝짓기

**TODO 1~2.** 주석을 따라 함수를 작성하세요.

💡 **힌트:** 위 표를 보세요.
- 요약문은 **검색**에 쓰입니다 → 어디에 넣어야 할까요?
- 원본은 **전달**에 쓰입니다 → 어디에 넣어야 할까요?

> ⚠️ **이 셀을 두 번 실행하면 데이터가 중복됩니다.** 다시 돌리려면 2번 셀부터 재실행하세요.

In [ ]:
# ── 3. 요약과 원본을 같은 ID로 짝짓기 ─────────────────────
# 1) langchain_core.documents 의 Document import

# 2) add_documents(retriever, doc_summaries, doc_contents) 함수 정의
#    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
#    - 원본 개수만큼 uuid4 로 doc_ids 생성 (문자열로 변환)
#    - 요약문마다 Document(page_content=요약, metadata={id_key: doc_ids[i]}) 생성

#    - TODO 1: 요약본은 어디에 넣을까요? (vectorstore / docstore)
#      → retriever.vectorstore.add_documents(...)

#    - TODO 2: 원본은 어디에 넣을까요?
#      → retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

# 3) 데이터가 있을 때만 각각 저장
#    - text_summaries  ↔ texts_2k_token
#    - table_summaries ↔ tables
#    - image_summaries ↔ img_base64_list

# 4) "색인 완료" 와 벡터 DB에 저장된 요약 수 출력 (vectorstore.get()["ids"] 의 길이)
# ⚠️ 이 셀을 두 번 실행하면 데이터가 중복됩니다.

## 4. 검색 테스트 — 요약으로 찾고 원본을 받는다

**TODO 3.** 본인 PDF에 맞는 질문을 적으세요.

💡 **힌트:** **그래프나 표를 봐야만** 답할 수 있는 질문이어야 합니다. 텍스트에도 똑같이 적혀 있는 내용을 물으면 나중에 Before/After 차이가 안 납니다.

> 좋은 예: "그래프에 나타난 2023년 추세를 설명해줘"
> 나쁜 예: "이 문서는 무엇에 관한 것인가요?"

In [ ]:
# ── 4. 검색 테스트 ────────────────────────────────────────
# TODO 3: 그래프나 표를 봐야만 답할 수 있는 질문을 query 에 적으세요

# 1) retriever.invoke(query) 결과를 found 에 저장
# 2) 검색된 원본 수 출력

# 3) found 를 순회하며
#    - content = doc.page_content (Document 가 아니면 doc 그대로)
#    - 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 에 맞으면 "이미지(Base64)", 아니면 "텍스트/표"
#    - 종류, 길이, 앞 120자(줄바꿈 제거) 출력

> ✅ **확인 포인트:** 검색 결과에 **이미지(Base64)가 최소 1건** 들어 있어야 합니다.
>
> 하나도 없다면 이미지 요약의 품질이 낮아 검색이 안 되는 것입니다. 6교시로 돌아가 프롬프트를 보강하세요.

> 💡 **여기가 오늘의 핵심 장면입니다.** 검색어와 글자가 하나도 안 겹치는 원본이 나왔죠? 요약문을 거쳐서 찾았기 때문입니다.

## 5. 검색 결과를 텍스트와 이미지로 분리

멀티모달 LLM에 넘기려면 둘을 다른 형식으로 포장해야 합니다. Base64 문자열의 **생김새**로 구분합니다.

아래 셀의 주석을 따라 코드를 작성하세요.

In [ ]:
# ── 5. 검색 결과를 텍스트와 이미지로 분리 ─────────────────
# 1) langchain_core.documents 의 Document import

# 2) split_image_text_types(docs) 함수 정의
#    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
#    - b64_images, texts_out 빈 리스트 준비
#    - Document 면 page_content 를 꺼내 문자열로 만들기
#    - Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
#      → 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 매칭 시 이미지, 아니면 텍스트
#    - {"images": b64_images, "texts": texts_out} 반환

# 3) 앞서 검색한 결과(found)로 동작 확인 → 이미지 / 텍스트 건수 출력

## 6. 최종 멀티모달 체인 만들기

체인의 구조는 세 칸입니다.

```
{검색 + 질문}  →  img_prompt_func  →  멀티모달 LLM  →  StrOutputParser
   재료 모으기      메시지로 조립        답변 생성        문자열로 꺼내기
```

**TODO 4~5.** 주석을 따라 체인을 작성하세요.

💡 **힌트**
- TODO 4: 6교시에서 Base64 앞에 붙였던 것과 같은 접두사입니다
- TODO 5: 위 그림의 3번 칸에 들어갈 변수 이름입니다

In [ ]:
# ── 6. 최종 멀티모달 체인 만들기 ──────────────────────────
# 1) import
#    - HumanMessage (langchain_core.messages)
#    - StrOutputParser (langchain_core.output_parsers)
#    - ChatPromptTemplate (langchain_core.prompts)
#    - RunnableLambda, RunnablePassthrough (langchain_core.runnables)
#    - ChatOpenAI (langchain_openai)

# 2) img_prompt_func(data_dict) 함수 정의
#    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
#    - messages 빈 리스트 준비
#    - 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
#         data_dict["context"]["images"] 마다 {"type": "image_url", "image_url": {"url": ...}} 추가
#         TODO 4: Base64 앞에 붙일 접두사를 완성하세요 (data:image/jpeg;base64,)
#    - 2) 텍스트 컨텍스트와 질문을 넣습니다
#         texts 를 줄바꿈으로 합치고 {"type": "text", "text": ...} 추가
#         지시문: "아래 자료를 근거로 질문에 한국어로 답하세요. 자료에 없는 내용은 모른다고 답하세요."
#         + 질문(data_dict["question"]) + 참고 텍스트
#    - [HumanMessage(content=messages)] 반환

# 3) 모델: ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)

# 4) 체인 조립 (LCEL)
#    {"context": retriever | RunnableLambda(split_image_text_types),
#     "question": RunnablePassthrough()}
#    | RunnableLambda(img_prompt_func)
#    | TODO 5: 멀티모달 LLM 변수
#    | StrOutputParser()

# 5) "최종 체인 조립 완료" 출력

## 7. 질의 실행

In [ ]:
# ── 7. 질의 실행 ──────────────────────────────────────────
# 1) 시작 시간 기록 (time.time())
# 2) chain.invoke(query) 결과를 answer 에 저장
# 3) 소요 시간, 질문(Q), 구분선, 답변 출력

> ✅ **확인 포인트:** 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있어야 합니다.
>
> 두루뭉술한 답이 나왔다면 질문을 더 구체적으로 바꿔보세요.

## 8. Before / After 비교 (가장 중요)

> 📌 **과제 채점 7점짜리 셀입니다. 반드시 하세요.**

같은 질문을 **텍스트 전용 RAG**와 **멀티모달 RAG**에 각각 던져 차이를 증거로 남깁니다.

먼저 비교용 텍스트 전용 RAG를 만듭니다. **이미지와 표를 뺀** 것 말고는 모든 조건이 같습니다.

In [ ]:
# ── 8-1. 비교용 텍스트 전용 RAG ────────────────────────────
# 이미지와 표를 뺀 것 말고는 모든 조건이 같아야 합니다.

# 1) 텍스트만 들어간 벡터 DB 생성
#    - Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))
#    - texts_2k_token 을 add_texts 로 저장
#    - as_retriever(search_kwargs={"k": 4}) 로 검색기 생성

# 2) 프롬프트: ChatPromptTemplate.from_template
#    - 멀티모달 체인과 같은 지시문 + {question} + {context}

# 3) 체인 조립
#    {"context": 검색기 | (문서들의 page_content 를 줄바꿈으로 합치는 함수),
#     "question": RunnablePassthrough()}
#    | 프롬프트 | model | StrOutputParser()

# 4) "비교용 텍스트 전용 RAG 준비 완료" 출력

In [ ]:
# ── 8-2. Before / After 출력 ───────────────────────────────
# 1) 구분선과 질문 출력

# 2) [BEFORE] 텍스트 전용 RAG
#    - text_only_chain.invoke(query) 결과를 before 에 저장 후 출력

# 3) [AFTER] 멀티모달 RAG
#    - chain.invoke(query) 결과를 after 에 저장 후 출력

# 4) "이 출력 전체를 캡처해 README.md에 넣으세요." 안내 출력

**TODO 6. 두 답변의 차이를 적으세요 (더블클릭해서 편집)**

- 텍스트 전용 RAG의 답변은 어땠나요:
- 멀티모달 RAG의 답변은 어땠나요:
- 차이가 생긴 이유는:

> 💡 **차이가 잘 안 드러난다면** 질문이 잘못된 것입니다. 텍스트에도 같은 내용이 적혀 있을 가능성이 큽니다. **그래프에만 있는 정보**를 묻는 질문으로 바꿔 4번 셀부터 다시 돌리세요.

> 📸 **지금 이 출력을 스크린샷으로 저장하세요.** 포트폴리오에서 가장 강력한 자료입니다.

## 9. 번역 체인 통합 (선택)

번역만 로컬 모델에 맡겨 비용을 줄이는 구성입니다.

1. Ollama 설치

설치 파일

https://ollama.com/download 버전별 설치 파일(OllamaSetup.exe)을 받습니다.
masOS 터미널 실행하기:curl -fsSL https://ollama.com/install.sh | sh
windows powershell 에서 실행하기 :irm https://ollama.com/install.ps1 | iex
설치가 끝나면 작업 표시줄 트레이에 라마 아이콘이 뜹니다.

설치 후 새 터미널을 열어 확인합니다.
ollama --version

2. llama3.1:8b 모델 받기
powershell에서 실행 
ollama pull llama3.1:8b

용량이 약 4.9GB라서 네트워크에 따라 몇 분 걸립니다.
메모리는 8GB 이상을 권장합니다. 16GB면 여유가 있습니다.

3. 동작 테스트
powershell
ollama run llama3.1:8b "안녕하세요를 영어로 번역해줘"

답이 나오면 성공입니다. 대화 모드에서는 /bye로 빠져나옵니다.

4. 파이썬 패키지 설치

VS Code 터미널에서 가상환경(.venv)이 켜진 상태로 설치합니다.
powershell
pip install langchain-ollama

5. 용량확보하기 위해서는 실습 테스트 후 지워주세요


In [ ]:
# ── 실습 후 용량 확보 (Ollama 정리) ───────────────────────
# 주석 해제 후 powershell 에서 실행
# ollama list              # 설치된 모델 확인
# ollama rm llama3.1:8b    # 삭제
# ollama list              # 목록에서 사라졌는지 확인

# 2. 파이썬 연동 패키지 삭제
# 가상환경(.venv)이 켜진 상태에서 실행합니다.
# %pip uninstall -y langchain-ollama ollama

# 삭제한 뒤에는 커널을 재시작하세요.

# 3. Ollama 프로그램까지 완전히 제거
# 설정 → 앱 → 설치된 앱 → Ollama → 제거를 선택합니다. winget uninstall Ollama.Ollama 로도 됩니다.
# 남은 모델 폴더를 지웁니다.

# powershell
# Remove-Item -Recurse -Force "$env:USERPROFILE\.ollama"

In [ ]:
# ── 9-1. 노트북에서 Ollama 연결 확인 ──────────────────────
# 1) langchain_ollama.llms 의 OllamaLLM import
# 2) OllamaLLM(model="llama3.1:8b") 생성
# 3) "Translate to Korean: Hello, world" 를 invoke 해서 결과 출력

In [ ]:
# ── 9-2. 번역 체인 통합 (선택) ────────────────────────────
# Ollama + llama3.1:8b 가 떠 있는 경우에만 실행하세요

# 1) 번역 프롬프트: ChatPromptTemplate.from_template
#    - "다음 영문 답변을 자연스러운 한국어로 번역하세요. 숫자와 고유명사는 그대로 두세요." + {answer}

# 2) translator = OllamaLLM(model="llama3.1:8b")

# 3) final_chain 조립
#    chain → (답변 문자열을 {"answer": 답변} 딕셔너리로 포장) → 번역 프롬프트 → translator → StrOutputParser()

# 4) final_chain.invoke(query) 결과 출력

---

## ✅ 체크포인트 4 — 모두 체크되면 완료

- [ ] 질문 하나에 대해 답변이 출력된다
- [ ] 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있다
- [ ] 검색 결과에 이미지(Base64)가 최소 1건 포함된다
- [ ] Before/After 출력을 **캡처해 저장**했다
- [ ] 질의응답 5건 마크다운 표를 복사해 두었다
- [ ] TODO 1~6을 모두 채웠다

## 생각해 볼 질문 (답을 적어보세요)

**Q1. 요약과 원본에 같은 `doc_id`를 붙이지 않으면 어떤 일이 벌어질까요?**

답:

**Q2. 검색은 요약으로 하는데, 왜 답변 생성에는 원본을 쓸까요? 요약만 쓰면 안 되나요?**

답:

**Q3. `InMemoryStore`는 커널을 끄면 사라집니다. 실제 서비스로 만들려면 무엇을 바꿔야 할까요?**

답:

## 자주 나는 에러

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `FileNotFoundError: summaries.pkl` |  저장 셀 미실행 | 앞 노트북 마지막 셀 재실행 |
| 검색 결과에 이미지가 없음 | 이미지 요약 품질이 낮음 | 프롬프트 보강 |
| 답변이 "자료에 없습니다" | 검색이 엉뚱한 원본을 가져옴 | 질문을 요약문 표현에 가깝게 수정 |
| 같은 원본이 여러 번 나옴 | `add_documents`를 두 번 실행 | 2번 셀부터 재실행 |
| `RateLimitError` | 질의 5건 연속 호출 | 잠시 대기 후 재실행 |
| Before/After 차이가 없음 | 질문이 텍스트로도 답 가능 | 그래프에만 있는 정보를 묻도록 수정 |
| `NameError: found` | 4번 셀을 안 돌림 | 위에서부터 순서대로 재실행 |

## 과제로 가져갈 것

| 산출물 | 어디서 나왔나 |
| --- | --- |
| Before/After 캡처 | 8번 셀 출력 |
| 질의응답 5건 표 | 10번 셀이 출력한 마크다운 |
| 아키텍처 다이어그램 | 직접 작도 (Excalidraw / Mermaid) |
| `README.md` | 문제 정의 + 위 자료들 |
| `.gitignore` | `.env`, `data/figures/`, `*.pdf` |

